# COMP9517 — E4 Checkpoint Inference / Evaluation Demo

This notebook is designed specifically for the **presentation code demonstration**.

It **does not retrain the model**.

The demo performs the following steps:

1. Connect to Google Drive;
2. Locate the saved E4 checkpoint and held-out test set;
3. Re-create the EfficientNet-B0 architecture;
4. Load the trained weights from the saved checkpoint;
5. Run a small inference example;
6. Run evaluation on the complete 5,000-image test set;
7. Report Top-1 accuracy, Top-5 accuracy, Macro Precision, Macro Recall, Macro-F1, and inference speed.

The model demonstrated here is:

**E4 — EfficientNet-B0, ImageNet pretrained, strong augmentation**


## 1. Mount Google Drive and import libraries

The checkpoint was saved during the original training experiment.  
For this demonstration, we only restore the saved model and run inference/evaluation.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import time

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

from sklearn.metrics import precision_recall_fscore_support

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Mounted at /content/drive
PyTorch version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/COMP9517/shared_dataset_seed9517.zip")
EXTRACT_DIR = Path("/content/dataset")

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Archive not found: {ZIP_PATH}\n"
        "Please check the shared-folder shortcut and file path."
    )

print(f"Archive found: {ZIP_PATH}")
print(f"Size: {ZIP_PATH.stat().st_size / 1024**3:.2f} GB")

marker = EXTRACT_DIR / ".extract_complete"
if not marker.exists():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    marker.write_text("complete", encoding="utf-8")
    print("Extraction complete.")
else:
    print("Already extracted in this session; skipping.")

print("\nTop-level directories:")
for item in sorted(EXTRACT_DIR.iterdir()):
    print("-", item)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive found: /content/drive/MyDrive/COMP9517/shared_dataset_seed9517.zip
Size: 2.52 GB
Extracting...
Extraction complete.

Top-level directories:
- /content/dataset/.extract_complete
- /content/dataset/shared_dataset_seed9517


## 2. Configure dataset and checkpoint paths

These paths match the paths used in the current COMP9517 deep-learning analysis / Grad-CAM notebook.

The demo uses the **held-out test set only**. No training images are required.


In [4]:
# ============================================================
# Paths
# ============================================================

RESULTS_DIR = Path(
    "/content/drive/MyDrive/COMP9517/deep_learning_results"
)

TEST_DIR = Path(
    "/content/dataset/shared_dataset_seed9517/images/test"
)

CHECKPOINT_PATH = (
    RESULTS_DIR / "E4_efficientnet_b0_pretrained_strong_best.pt"
)

print("RESULTS_DIR    :", RESULTS_DIR)
print("TEST_DIR       :", TEST_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)

print("\nPath checks:")
print("Results directory exists:", RESULTS_DIR.exists())
print("Test directory exists   :", TEST_DIR.exists())
print("Checkpoint exists       :", CHECKPOINT_PATH.exists())

if not TEST_DIR.exists():
    raise FileNotFoundError(
        f"Test directory not found: {TEST_DIR}\n"
        "Run the dataset extraction/setup step before this demo."
    )

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}\n"
        "Check that Google Drive is mounted and the checkpoint path is correct."
    )

print("\nAll required files are available.")


RESULTS_DIR    : /content/drive/MyDrive/COMP9517/deep_learning_results
TEST_DIR       : /content/dataset/shared_dataset_seed9517/images/test
CHECKPOINT_PATH: /content/drive/MyDrive/COMP9517/deep_learning_results/E4_efficientnet_b0_pretrained_strong_best.pt

Path checks:
Results directory exists: True
Test directory exists   : True
Checkpoint exists       : True

All required files are available.


## 3. Build the test DataLoader

The evaluation preprocessing is the same as in the original deep-learning experiment:

- Resize to 256;
- Center crop to 224 × 224;
- Convert to tensor;
- Apply ImageNet normalization.

**No random augmentation is used during testing.**


In [5]:
# ============================================================
# Evaluation preprocessing
# ============================================================

IMAGE_SIZE = 224
BATCH_SIZE = 64

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

NUM_CLASSES = len(test_dataset.classes)

print("Number of classes    :", NUM_CLASSES)
print("Number of test images:", len(test_dataset))
print("Batch size           :", BATCH_SIZE)
print("First 10 class folders:")
print(test_dataset.classes[:10])

if NUM_CLASSES != 500:
    print(
        f"\nWarning: expected 500 classes, but found {NUM_CLASSES}. "
        "Please verify TEST_DIR before presenting."
    )

if len(test_dataset) != 5000:
    print(
        f"\nWarning: expected 5,000 test images, but found {len(test_dataset)}. "
        "Please verify TEST_DIR before presenting."
    )


Number of classes    : 500
Number of test images: 5000
Batch size           : 64
First 10 class folders:
['000', '001', '002', '003', '004', '005', '006', '007', '008', '009']


## 4. Re-create E4 and load the saved checkpoint

This is the key checkpoint demonstration.

`weights=None` here does **not** mean that E4 is being trained from scratch again.  
It only creates an empty EfficientNet-B0 architecture with the correct structure.

The trained parameters are then restored using:

```python
model.load_state_dict(state_dict)
```

No optimizer, loss function, backpropagation, or training loop is executed in this notebook.


In [6]:
# ============================================================
# Re-create E4 architecture
# ============================================================

model = models.efficientnet_b0(weights=None)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES,
)

# ============================================================
# Load saved checkpoint
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
)

state_dict = (
    checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
    else checkpoint
)

model.load_state_dict(state_dict)

model = model.to(DEVICE)
model.eval()

print("Checkpoint loaded successfully.")
print("Model          : EfficientNet-B0")
print("Output classes :", NUM_CLASSES)
print("Mode           : evaluation (model.eval())")

if isinstance(checkpoint, dict):
    if "epoch" in checkpoint:
        print("Saved epoch    :", checkpoint["epoch"])

    if "best_val_macro_f1" in checkpoint:
        print(
            "Best val Macro-F1:",
            checkpoint["best_val_macro_f1"]
        )


Checkpoint loaded successfully.
Model          : EfficientNet-B0
Output classes : 500
Mode           : evaluation (model.eval())
Saved epoch    : 12
Best val Macro-F1: 0.7085023502568357


## 5. Quick inference demonstration

This cell takes **one batch of test images** and sends it through the restored checkpoint.

For a batch size of 64:

- Input shape should be approximately `[64, 3, 224, 224]`;
- Model output shape should be approximately `[64, 500]`.

The 500 output values for each image are the class scores produced by the model.


In [7]:
# ============================================================
# One-batch inference
# ============================================================

images, targets = next(iter(test_loader))

images = images.to(DEVICE)
targets = targets.to(DEVICE)

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=torch.cuda.is_available(),
    ):
        logits = model(images)

predictions = logits.argmax(dim=1)

print("Input batch shape :", tuple(images.shape))
print("Model output shape:", tuple(logits.shape))

print("\nFirst 10 predictions:")
for i in range(min(10, len(predictions))):
    true_idx = targets[i].item()
    pred_idx = predictions[i].item()

    true_class = test_dataset.classes[true_idx]
    pred_class = test_dataset.classes[pred_idx]

    status = "CORRECT" if true_idx == pred_idx else "WRONG"

    print(
        f"{i + 1:2d}. "
        f"True: {true_class} | "
        f"Predicted: {pred_class} | "
        f"{status}"
    )


Input batch shape : (64, 3, 224, 224)
Model output shape: (64, 500)

First 10 predictions:
 1. True: 000 | Predicted: 016 | WRONG
 2. True: 000 | Predicted: 000 | CORRECT
 3. True: 000 | Predicted: 000 | CORRECT
 4. True: 000 | Predicted: 080 | WRONG
 5. True: 000 | Predicted: 027 | WRONG
 6. True: 000 | Predicted: 001 | WRONG
 7. True: 000 | Predicted: 000 | CORRECT
 8. True: 000 | Predicted: 000 | CORRECT
 9. True: 000 | Predicted: 000 | CORRECT
10. True: 000 | Predicted: 001 | WRONG


## 6. Full held-out test-set evaluation

This cell runs the restored E4 model over the complete test set.

It calculates the same main metrics reported in the project:

- Top-1 Accuracy;
- Top-5 Accuracy;
- Macro Precision;
- Macro Recall;
- Macro-F1.

The report values for E4 were approximately:

- **Top-1: 72.16%**
- **Top-5: 89.86%**
- **Macro-F1: 71.77%**

The newly computed values should match or be extremely close when the same checkpoint, dataset, class mapping, and preprocessing are used.


In [8]:
@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    all_targets = []
    all_predictions = []

    top1_correct = 0
    top5_correct = 0
    total = 0

    start = time.perf_counter()

    for images, targets in loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available(),
        ):
            logits = model(images)

        predictions = logits.argmax(dim=1)

        top5_predictions = logits.topk(
            k=min(5, logits.size(1)),
            dim=1,
        ).indices

        top1_correct += (
            predictions == targets
        ).sum().item()

        top5_correct += (
            top5_predictions
            .eq(targets.view(-1, 1))
            .any(dim=1)
            .sum()
            .item()
        )

        total += images.size(0)

        all_targets.extend(targets.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())

    elapsed = time.perf_counter() - start

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_targets,
            all_predictions,
            average="macro",
            zero_division=0,
        )
    )

    return {
        "top1_accuracy": top1_correct / total,
        "top5_accuracy": top5_correct / total,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "inference_seconds": elapsed,
        "images_per_second": total / elapsed,
        "num_test_images": total,
    }


metrics = evaluate_model(model, test_loader)

print("=" * 58)
print("E4 CHECKPOINT — HELD-OUT TEST EVALUATION")
print("=" * 58)
print(f"Test images       : {metrics['num_test_images']}")
print(f"Top-1 Accuracy    : {metrics['top1_accuracy'] * 100:.2f}%")
print(f"Top-5 Accuracy    : {metrics['top5_accuracy'] * 100:.2f}%")
print(f"Macro Precision   : {metrics['macro_precision'] * 100:.2f}%")
print(f"Macro Recall      : {metrics['macro_recall'] * 100:.2f}%")
print(f"Macro-F1          : {metrics['macro_f1'] * 100:.2f}%")
print(f"Inference time    : {metrics['inference_seconds']:.2f} s")
print(f"Images / second   : {metrics['images_per_second']:.2f}")
print("=" * 58)

print("\nDemo complete: checkpoint successfully loaded and evaluated.")


E4 CHECKPOINT — HELD-OUT TEST EVALUATION
Test images       : 5000
Top-1 Accuracy    : 72.16%
Top-5 Accuracy    : 89.86%
Macro Precision   : 74.38%
Macro Recall      : 72.16%
Macro-F1          : 71.77%
Inference time    : 34.48 s
Images / second   : 145.00

Demo complete: checkpoint successfully loaded and evaluated.


## Presentation summary

During the video, this notebook demonstrates the following complete runtime path:

```text
Saved E4 checkpoint
        ↓
Re-create EfficientNet-B0
        ↓
Load trained state_dict
        ↓
model.eval()
        ↓
Held-out test images
        ↓
Forward inference
        ↓
Predictions
        ↓
Top-1 / Top-5 / Macro-F1
```

The important point is that **the training stage is not repeated**.  
The video demonstrates that the software can successfully restore the previously trained model and use it for real inference/evaluation.
